In [1]:
import os
import ee
import geemap

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import math
import geopandas as gpd

import plotly.express as px
import plotly.graph_objects as go


import random
from shapely.geometry import Polygon, mapping

In [2]:
m = geemap.Map(center=[40,-100], zoom=4)
m

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

In [14]:
df_markom = pd.read_csv('HoBo/markom_data_results/model_data.csv')
df_markom.head()

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B
0,1/27/2024,talber360_downstream,1.835000,0.10,0.07500,56.3,1.912847,1/27/2024,7.312941,84.12,11.62,4.26
1,1/27/2024,bosque_ref,3.345000,0.10,0.07500,56.3,1.912847,1/27/2024,3.434193,81.34,18.66,0.00
2,1/27/2024,talbert_impact,1.690000,0.10,0.07500,56.3,1.912847,1/27/2024,3.987288,87.91,10.06,2.03
3,2/11/2024,talber360_downstream,3.283333,1.37,1.25125,54.7,2.382639,2/11/2024,7.312941,84.12,11.62,4.26
4,2/11/2024,bosque_ref,15.283333,1.37,1.25125,54.7,2.382639,2/11/2024,3.434193,81.34,18.66,0.00


In [15]:
# Load brushy shapefile into a GeoDataFrame
bosque = geemap.shp_to_ee("china_spring/china_spring_ranch.shp")
mcm = geemap.shp_to_ee("mcmanamay_ranch/manamay_ranch.shp")
talbert = geemap.shp_to_ee("talbert_ranch/talbert_ranch.shp")
talbert360 = geemap.shp_to_ee("talbert_360/talbert_360.shp")

In [16]:
## *
# Function to mask clouds using the Sentinel-2 QA band
# @param {ee.Image} image Sentinel-2 image
# @return {ee.Image} cloud masked Sentinel-2 image
#
map = geemap.Map(center=[40,-100], zoom=4)

roi = ee.Geometry.BBox(-97.528381, 31.781736, -96.630249, 31.292487)
aoi = ee.Geometry.Rectangle([-97.021749, 31.653679, -96.581087, 31.365537])

def maskS2clouds(image):
    qa = image.select('QA60')

    # Bits 10 and 11 are clouds and cirrus, respectively.
    cloudBitMask = 1 << 10
    cirrusBitMask = 1 << 11

    # Both flags should be set to zero, indicating clear conditions.
    mask = qa.bitwiseAnd(cloudBitMask).eq(0) \
    .And(qa.bitwiseAnd(cirrusBitMask).eq(0))

    return image.updateMask(mask).divide(10000)
    
# .filterDate('2024-09-01', '2024-10-01') \3/30/2024

S2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED") \
.filterDate('2024-05-11', '2024-06-20') \
.filterBounds(aoi) \
.filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',2)) \
.map(maskS2clouds)

visualization = {
    'min': 0.0,
    'max': 0.3,
    'bands': ['B4', 'B3', 'B2'],
}


style1 = {'color': 'ffffff', 'fillColor': '00000000'}

map.addLayer(S2.median(), visualization, 'RGB')
map.addLayer(roi, {}, 'roi', shown=False)
map.addLayer(bosque, {}, 'bosque', shown=True)
map.addLayer(talbert360, {}, 'talbert', shown=True)
map.centerObject(aoi)
map

Map(center=[40, -100], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(ch…

In [17]:
S2.size()

In [18]:
# SS2 = S2.median()
# nir = SS2.select('B8')
# red = SS2.select('B4')
# ndvi = nir.subtract(red).divide(nir.add(red)).rename('NDVI')

def ndvi_img(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('ndvi')
    return ndvi \
    .copyProperties(image, ['system:time_start', 'system:time_end'])


ndvi = S2.map(ndvi_img)


Map = geemap.Map()


ndviparam = {
    'min': -1,
    'max': 1,
    'palette': ['blue', 'white', 'green']
}

median_ndvi = ndvi.median()

Map.addLayer(median_ndvi, ndviparam, 'ndvi')
Map.addLayer(bosque, {}, 'bosque', shown=True)
Map.addLayer(talbert360, {}, 'talbert', shown=True)
#Map.addLayer(fishnet.style(**style), {}, 'Fishnet')
Map.centerObject(aoi)
Map

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [19]:
df_markom['Site'].unique()

array(['talber360_downstream', 'bosque_ref', 'talbert_impact', 'mcm_ref1'],
      dtype=object)

In [20]:
site_polygons = {
    'talber360_downstream': talbert360,
    'talbert_impact': talbert,
    'bosque_ref': bosque,
    'mcm_ref1': mcm
}

In [21]:
# Example: checking 'Site' column values against dictionary keys
missing_keys = set(df_markom['Site'].unique()) - set(site_polygons.keys())

# Print missing keys (if any)
print("Missing keys:", missing_keys)

Missing keys: set()


In [22]:
all_keys_present = set(df_markom['Site'].unique()).issubset(site_polygons.keys())
print(all_keys_present)  # True if all are found in the dict keys

True


In [23]:
df_markom['Date'].unique()

array(['1/27/2024', '2/11/2024', '2/12/2024', '2/13/2024', '2/18/2024',
       '3/7/2024', '3/8/2024', '3/23/2024', '3/25/2024', '3/30/2024',
       '4/21/2024', '4/28/2024', '4/30/2024', '5/2/2024', '5/20/2024',
       '5/22/2024', '5/24/2024', '5/30/2024', '5/31/2024', '6/10/2024',
       '6/12/2024', '9/17/2024', '9/19/2024', '11/5/2024', '1/30/2025',
       '3/4/2025', '4/24/2025'], dtype=object)

In [24]:
df_markom['Date'].value_counts()

Date
2/11/2024    4
3/25/2024    4
4/28/2024    4
5/2/2024     4
3/8/2024     3
3/7/2024     3
1/30/2025    3
3/4/2025     3
4/24/2025    3
1/27/2024    3
4/21/2024    2
11/5/2024    2
5/31/2024    2
5/20/2024    2
5/24/2024    2
2/12/2024    1
3/23/2024    1
2/18/2024    1
2/13/2024    1
3/30/2024    1
5/30/2024    1
4/30/2024    1
5/22/2024    1
9/19/2024    1
9/17/2024    1
6/12/2024    1
6/10/2024    1
Name: count, dtype: int64

In [25]:
# Find dates that appear only once
unique_dates = df_markom['Date'].value_counts()[df_markom['Date'].value_counts() == 1].index
df_markom[df_markom['Date'].isin(unique_dates)]

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B
7,2/12/2024,mcm_ref1,0.419333,1.37,0.399583,59.4,2.845486,2/12/2024,0.082201,100.00,0.00,0.00
8,2/13/2024,mcm_ref1,0.471000,1.37,0.399583,66.4,1.120139,2/12/2024,0.082201,100.00,0.00,0.00
9,2/18/2024,mcm_ref1,0.398000,1.37,0.399583,54.7,0.754861,2/12/2024,0.082201,100.00,0.00,0.00
16,3/23/2024,mcm_ref1,0.056000,1.32,1.145934,68.2,2.069231,3/22/2024,0.082201,100.00,0.00,0.00
21,3/30/2024,mcm_ref1,0.280000,0.12,0.099271,80.2,3.131944,3/28/2024,0.082201,100.00,0.00,0.00
28,4/30/2024,talbert_impact,2.501667,0.88,0.293333,85.6,1.633681,4/29/2024,3.987288,87.91,10.06,2.03
35,5/22/2024,talbert_impact,4.720000,0.48,0.112014,90.7,2.185069,5/22/2024,3.987288,87.91,10.06,2.03
38,5/30/2024,talbert_impact,8.585000,1.71,0.352951,90.7,1.087153,5/30/2024,3.987288,87.91,10.06,2.03
41,6/10/2024,talbert_impact,1.215000,1.51,0.508576,93.6,0.327083,6/6/2024,3.987288,87.91,10.06,2.03
42,6/12/2024,talbert_impact,6.706667,0.85,0.564757,89.1,0.653125,6/12/2024,3.987288,87.91,10.06,2.03


In [26]:
unique_dates = df_markom['Date'].value_counts()[df_markom['Date'].value_counts() == 2].index
df_markom[df_markom['Date'].isin(unique_dates)]

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B
22,4/21/2024,talber360_downstream,3.153333,0.01,0.000556,69.1,1.109722,4/20/2024,7.312941,84.12,11.62,4.26
23,4/21/2024,talbert_impact,4.206667,0.01,0.000556,69.1,1.109722,4/20/2024,3.987288,87.91,10.06,2.03
33,5/20/2024,bosque_ref,4.068889,2.06,1.630833,90.1,2.231250,5/17/2024,3.434193,81.34,18.66,0.00
34,5/20/2024,talbert_impact,3.151667,2.06,1.630833,90.1,2.231250,5/17/2024,3.987288,87.91,10.06,2.03
36,5/24/2024,bosque_ref,6.550000,0.74,0.432188,90.0,1.783681,5/24/2024,3.434193,81.34,18.66,0.00
37,5/24/2024,talbert_impact,3.400000,0.74,0.432188,90.0,1.783681,5/24/2024,3.987288,87.91,10.06,2.03
39,5/31/2024,bosque_ref,5.350000,3.92,3.711597,84.2,1.096528,5/31/2024,3.434193,81.34,18.66,0.00
40,5/31/2024,talbert_impact,31.500000,3.92,3.711597,84.2,1.096528,5/31/2024,3.987288,87.91,10.06,2.03
45,11/5/2024,talber360_downstream,6.060000,1.70,1.693160,71.1,0.664583,11/5/2024,7.312941,84.12,11.62,4.26
46,11/5/2024,talbert_impact,2.728500,1.70,1.693160,71.1,0.664583,11/5/2024,3.987288,87.91,10.06,2.03


In [28]:
#combined_df = combined_df[combined_df['Date'] != '1/22/2024']

In [510]:
df_markom[df_markom['Date'] == '5/30/2024']

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B,NDVI
38,5/30/2024,talbert_impact,8.585,1.71,0.352951,90.7,1.087153,5/30/2024,3.987288,87.91,10.06,2.03,None


In [511]:
#target_date = ['2/10/2024', '2/12/2024']
target_date = '5/30/2024'

In [512]:
#combined_df[combined_df['Date'].isin(target_date)]
df_markom[df_markom['Date'] == target_date]

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B,NDVI
38,5/30/2024,talbert_impact,8.585,1.71,0.352951,90.7,1.087153,5/30/2024,3.987288,87.91,10.06,2.03,None


In [513]:
# Filter combined_df by the target date
filtered_df = df_markom[df_markom['Date'] == target_date].copy()
#filtered_df = combined_df[combined_df['Date'].isin(target_date)].copy()

# Ensure the NDVI column exists
if 'NDVI' not in df_markom.columns:
    df_markom['NDVI'] = None

# Loop over the filtered rows by site
for idx, row in filtered_df.iterrows():
    site_name = row['Site']
    if site_name in site_polygons:
        try:
            feature = site_polygons[site_name]
            
            # Reduce the NDVI raster over the site polygon (mean NDVI)
            mean_dict = median_ndvi.reduceRegion(
                reducer=ee.Reducer.mean(),
                geometry=feature.geometry(),
                scale=10,
                maxPixels=1e9,    # Increased to 9 billion pixels
                bestEffort=True   # Allow Earth Engine to adjust the scale if needed
            ).getInfo()
            
            mean_ndvi = mean_dict.get('ndvi')  # or replace 'NDVI' with actual band name

            # Insert NDVI value into original DataFrame at correct row
            df_markom.at[idx, 'NDVI'] = mean_ndvi

        except Exception as e:
            print(f"Error extracting NDVI for site {site_name}: {e}")

In [514]:
mean_dict

{'ndvi': 0.4203991507519725}

In [515]:
#combined_df[combined_df['Date'].isin(target_date)]
df_markom[df_markom['Date'] == target_date]

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B,NDVI
38,5/30/2024,talbert_impact,8.585,1.71,0.352951,90.7,1.087153,5/30/2024,3.987288,87.91,10.06,2.03,0.420399


In [516]:
df_markom.head(10)

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B,NDVI
0,1/27/2024,talber360_downstream,1.835000,0.10,0.075000,56.3,1.912847,1/27/2024,7.312941,84.12,11.62,4.26,0.490855
1,1/27/2024,bosque_ref,3.345000,0.10,0.075000,56.3,1.912847,1/27/2024,3.434193,81.34,18.66,0.00,0.407345
2,1/27/2024,talbert_impact,1.690000,0.10,0.075000,56.3,1.912847,1/27/2024,3.987288,87.91,10.06,2.03,0.483244
3,2/11/2024,talber360_downstream,3.283333,1.37,1.251250,54.7,2.382639,2/11/2024,7.312941,84.12,11.62,4.26,0.529156
4,2/11/2024,bosque_ref,15.283333,1.37,1.251250,54.7,2.382639,2/11/2024,3.434193,81.34,18.66,0.00,0.453886
5,2/11/2024,mcm_ref1,0.804667,1.37,1.251250,54.7,2.382639,2/11/2024,0.082201,100.00,0.00,0.00,0.589544
6,2/11/2024,talbert_impact,3.411667,1.37,1.251250,54.7,2.382639,2/11/2024,3.987288,87.91,10.06,2.03,0.526529
7,2/12/2024,mcm_ref1,0.419333,1.37,0.399583,59.4,2.845486,2/12/2024,0.082201,100.00,0.00,0.00,0.589544
8,2/13/2024,mcm_ref1,0.471000,1.37,0.399583,66.4,1.120139,2/12/2024,0.082201,100.00,0.00,0.00,0.589544
9,2/18/2024,mcm_ref1,0.398000,1.37,0.399583,54.7,0.754861,2/12/2024,0.082201,100.00,0.00,0.00,0.589544


In [517]:
df_markom.tail()

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B,NDVI
51,3/4/2025,bosque_ref,2.220000,1.49,0.021562,70.9,7.317014,3/4/2025,3.434193,81.34,18.66,0.00,0.394601
52,3/4/2025,talbert_impact,71.466667,1.49,0.021562,70.9,7.317014,3/4/2025,3.987288,87.91,10.06,2.03,0.519238
53,4/24/2025,talber360_downstream,93.966667,0.50,0.006763,76.1,3.105314,4/24/2025,7.312941,84.12,11.62,4.26,0.629272
54,4/24/2025,bosque_ref,6.780000,0.50,0.006763,76.1,3.105314,4/24/2025,3.434193,81.34,18.66,0.00,0.620123
55,4/24/2025,talbert_impact,28.450000,0.50,0.006763,76.1,3.105314,4/24/2025,3.987288,87.91,10.06,2.03,0.640406


In [521]:
df_markom['NDVI'].isnull().any()

False

In [522]:
df_markom['NDVI'].isnull().sum()

0

In [523]:
df_markom.head(10)

,Date,Site,Turbidity,max_rain_event,mean_rain_event,max_air_temp,mean_wind_gust,rain_event_date,drainage_area_km2,pct_Soil_HSG_D,pct_Soil_HSG_C,pct_Soil_HSG_B,NDVI
0,1/27/2024,talber360_downstream,1.835000,0.10,0.075000,56.3,1.912847,1/27/2024,7.312941,84.12,11.62,4.26,0.490855
1,1/27/2024,bosque_ref,3.345000,0.10,0.075000,56.3,1.912847,1/27/2024,3.434193,81.34,18.66,0.00,0.407345
2,1/27/2024,talbert_impact,1.690000,0.10,0.075000,56.3,1.912847,1/27/2024,3.987288,87.91,10.06,2.03,0.483244
3,2/11/2024,talber360_downstream,3.283333,1.37,1.251250,54.7,2.382639,2/11/2024,7.312941,84.12,11.62,4.26,0.529156
4,2/11/2024,bosque_ref,15.283333,1.37,1.251250,54.7,2.382639,2/11/2024,3.434193,81.34,18.66,0.00,0.453886
5,2/11/2024,mcm_ref1,0.804667,1.37,1.251250,54.7,2.382639,2/11/2024,0.082201,100.00,0.00,0.00,0.589544
6,2/11/2024,talbert_impact,3.411667,1.37,1.251250,54.7,2.382639,2/11/2024,3.987288,87.91,10.06,2.03,0.526529
7,2/12/2024,mcm_ref1,0.419333,1.37,0.399583,59.4,2.845486,2/12/2024,0.082201,100.00,0.00,0.00,0.589544
8,2/13/2024,mcm_ref1,0.471000,1.37,0.399583,66.4,1.120139,2/12/2024,0.082201,100.00,0.00,0.00,0.589544
9,2/18/2024,mcm_ref1,0.398000,1.37,0.399583,54.7,0.754861,2/12/2024,0.082201,100.00,0.00,0.00,0.589544


In [526]:
# df_markom['NDVI'] = pd.to_numeric(df_markom['NDVI'], errors='coerce')
# df_markom.to_csv('HoBo/markom_data_results/markom_model_database.csv', index=False, float_format='%.10f')